In [1]:
import pandas as pd

df = pd.read_csv('../Data/PS_20174392719_1491204439457_log.csv')
df.shape

(6362620, 11)

In [2]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [3]:
df['isFraud'].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [4]:
pd.crosstab(df['type'], df['isFraud'])

isFraud,0,1
type,,
CASH_IN,1399284,0
CASH_OUT,2233384,4116
DEBIT,41432,0
PAYMENT,2151495,0
TRANSFER,528812,4097


In [5]:
df.groupby('isFraud')['amount'].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,1.781970e+05,5.962370e+05,0.01,13368.395,74684.72,208364.76,92445516.64
1,8213.0,1.467967e+06,2.404253e+06,0.00,127091.330,441423.44,1517771.48,10000000.00


In [6]:
df[df['isFraud'] == 1][['amount', 'oldbalanceOrg', 'newbalanceOrig']].describe()

,amount,oldbalanceOrg,newbalanceOrig
count,8.213000e+03,8.213000e+03,8.213000e+03
mean,1.467967e+06,1.649668e+06,1.923926e+05
std,2.404253e+06,3.547719e+06,1.965666e+06
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.270913e+05,1.258224e+05,0.000000e+00
50%,4.414234e+05,4.389835e+05,0.000000e+00
75%,1.517771e+06,1.517771e+06,0.000000e+00
max,1.000000e+07,5.958504e+07,4.958504e+07


In [7]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [8]:
summary = """
# PaySim Dataset — Exploration Summary

**Dataset**: 6,362,620 transactions, 11 columns (step, type, amount, nameOrig,
oldbalanceOrg, newbalanceOrig, nameDest, oldbalanceDest, newbalanceDest,
isFraud, isFlaggedFraud). No missing values.

## Class Imbalance
- Genuine: 6,354,407 (99.871%)
- Fraud: 8,213 (0.129%) — even rarer than the credit card dataset (0.173%)

## Transaction Type is a Strong Signal
Fraud occurs ONLY in CASH_OUT (4,116 cases) and TRANSFER (4,097 cases) —
zero fraud in CASH_IN, DEBIT, or PAYMENT. Confirmed directly via crosstab.

## Amount Patterns
- Fraud mean amount (1,467,967) is ~8x higher than genuine (178,197) —
  opposite pattern from the credit card dataset, where fraud skewed smaller.

## Balance Patterns — Strong, Possibly Simulator-Specific Signal
- newbalanceOrig for fraud transactions: 0.00 at the 25th, 50th, AND 75th
  percentiles — the sender's account is drained to exactly zero in at least
  75% of fraud cases.
- This is an unusually clean signal, likely reflecting how PaySim's fraud
  injection logic works rather than universal real-world fraud behavior.
  Worth treating with some caution as a feature, and worth noting explicitly
  as a dataset characteristic.

## Implications for Model Training
- nameOrig/nameDest are IDs, not usable features — will be dropped.
- 'type' is categorical — needs encoding (one-hot).
- Given 6.3M rows, will train on a stratified sample for practical runtime.
- Class imbalance still requires SMOTE/class-weighting, same as before.
"""

with open('../notes/paysim_summary.md', 'w') as f:
    f.write(summary)

print("Saved.")

Saved.
